# Thử nghiệm Huấn luyện & Đánh giá kết hợp CNN-SVM

Notebook này đóng vai trò như môi trường thử nghiệm nhanh từng bước trong mô hình phân loại tin nhắn rác kết hợp CNN và SVM.

In [ ]:
import sys
import os
import pandas as pd
import numpy as np

# Thêm đường dẫn src
sys.path.append('../src')

## 1. Load Cấu hình & Dữ liệu đã chia tách

In [ ]:
from utils import load_config, get_logger

config = load_config("../config.yaml")
split_dir = "../" + config["paths"]["split_dir"]

train_df = pd.read_csv(os.path.join(split_dir, "train.csv"))
val_df = pd.read_csv(os.path.join(split_dir, "val.csv"))
test_df = pd.read_csv(os.path.join(split_dir, "test.csv"))

train_df['clean_text'] = train_df['clean_text'].fillna('')
val_df['clean_text'] = val_df['clean_text'].fillna('')
test_df['clean_text'] = test_df['clean_text'].fillna('')

print(f"Tập Train: {len(train_df)} mẫu")
print(f"Tập Val: {len(val_df)} mẫu")
print(f"Tập Test: {len(test_df)} mẫu")

## 2. Tokenize văn bản

In [ ]:
from feature_extraction import TextTokenizer

vocab_size = config["preprocessing"]["vocab_size"]
max_len = config["preprocessing"]["max_len"]

tokenizer = TextTokenizer(vocab_size=vocab_size, max_len=max_len)
tokenizer.fit(train_df['clean_text'].tolist())

X_train = tokenizer.texts_to_sequences(train_df['clean_text'].tolist())
X_val = tokenizer.texts_to_sequences(val_df['clean_text'].tolist())
X_test = tokenizer.texts_to_sequences(test_df['clean_text'].tolist())

y_train = train_df['label_code'].values
y_val = val_df['label_code'].values
y_test = test_df['label_code'].values

## 3. Khởi dựng và Huấn luyện CNN sơ bộ

In [ ]:
from cnn_model import build_cnn_model
from tensorflow.keras.callbacks import EarlyStopping

cnn_model = build_cnn_model(
    vocab_size=vocab_size,
    embedding_dim=config["cnn"]["embedding_dim"],
    max_len=max_len,
    num_filters=config["cnn"]["num_filters"],
    kernel_size=config["cnn"]["kernel_size"],
    dense_units=config["cnn"]["dense_units"],
    dropout_rate=config["cnn"]["dropout_rate"],
    learning_rate=config["cnn"]["learning_rate"]
)

early_stopping = EarlyStopping(monitor='val_loss', patience=2, restore_best_weights=True)

history = cnn_model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=5, # Chạy thử nghiệm nhanh 5 epochs
    batch_size=config["cnn"]["batch_size"],
    callbacks=[early_stopping]
)

## 4. Trích xuất đặc trưng & Huấn luyện SVM

In [ ]:
from tensorflow.keras.models import Model
from svm_model import SVMClassifier

# Trích xuất từ tầng dense_features của CNN
feature_extractor = Model(inputs=cnn_model.input, outputs=cnn_model.get_layer("dense_features").output)

X_train_features = feature_extractor.predict(X_train)
X_test_features = feature_extractor.predict(X_test)

# Huấn luyện SVM
svm_clf = SVMClassifier(C=config["svm"]["C"], kernel=config["svm"]["kernel"], gamma=config["svm"]["gamma"])
svm_clf.fit(X_train_features, y_train)

## 5. Đánh giá nhanh kết quả

In [ ]:
from sklearn.metrics import classification_report, accuracy_score

y_pred_svm = svm_clf.predict(X_test_features)
print(f"Độ chính xác SVM: {accuracy_score(y_test, y_pred_svm):.4%}")
print(classification_report(y_test, y_pred_svm, target_names=['Ham', 'Spam']))

## 6. Chạy dự thử đoán trực tiếp

In [ ]:
from preprocessing import clean_text

test_sms = [
    "Hey, are you free tonight? Let's hang out!",
    "URGENT: Your mobile number has been selected for a £2000 prize! Call 09061111111 immediately."
]

for sms in test_sms:
    cleaned = clean_text(sms)
    seq = tokenizer.texts_to_sequences([cleaned])
    feats = feature_extractor.predict(seq)
    label_pred = svm_clf.predict(feats)[0]
    probs = svm_clf.predict_proba(feats)[0]
    
    print(f"Tin nhắn: '{sms}'")
    print(f"Kết quả dự đoán: {'SPAM' if label_pred == 1 else 'HAM'} (Xác suất Spam: {probs[1]:.2%})")
    print("-"*50)